# Phase 02 - Data

Dataset index, EXIF-correct image IO, resolution cache, near-duplicate audit,
stratified group-aware splits and dataset figures. See `docs/instructions/02-data.md`.

Cells whose first line is `#| export` are copied, in order, into the
generated `group3-starvenus.py` by `tools/export_py.py`. Keep outputs cleared before committing.

Everything this phase writes lives under `<data-root>/_work/`; the dataset files are never touched.

## Constants and label mapping (single source of truth)

Labels come from the **filename prefix** and are cross-checked against the CSV `Diseases` column.
Folder names are never used (they contain typos and trailing spaces).

In [ ]:
#| export
import csv
import hashlib
import json
import multiprocessing
import os
import struct
import sys
import tempfile
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import cv2
import numpy as np

IMAGE_SUFFIXES = frozenset({".jpg", ".jpeg"})
WORK_DIRNAME = "_work"  # everything the code generates lives in <data_root>/_work/

# Fixed class order, used in every confusion matrix, table and figure.
CLASS_NAMES = ("Brown Spot", "Leaf Scald", "Rice Blast", "Rice Tungro", "Sheath Blight")
DOMAINS = ("white", "field")

# (filename prefix, class, domain, exact string in the CSV `Diseases` column)
LABEL_TABLE = (
    ("bs_wb_", "Brown Spot", "white", "Brown Spot(white Background)"),
    ("ls_wb_", "Leaf Scald", "white", "Leaf Scaled(white Background)"),
    ("rb_wb_", "Rice Blast", "white", "Rice Blast(white Background)"),
    ("rt_wb_", "Rice Tungro", "white", "Rice Tungro(white Background)"),
    ("sb_wb_", "Sheath Blight", "white", "Shath Blight(white Background)"),
    ("bsf_", "Brown Spot", "field", "Browon Spot(Feild Background)"),
    ("lsf_", "Leaf Scald", "field", "Leaf Scaled(Feild Background)"),
    ("rbf_", "Rice Blast", "field", "Rice Blast(Feild Background)"),
    ("rtf_", "Rice Tungro", "field", "Rice Turgro(Feild Background)"),
    ("sbf_", "Sheath Blight", "field", "Sheath Blight(Feild Background)"),
)
# Longest prefix first, so that a prefix that extends another one always wins.
PREFIXES_LONGEST_FIRST = tuple(sorted(LABEL_TABLE, key=lambda row: -len(row[0])))
CSV_LABELS = {csv_label: (cls, domain) for _, cls, domain, csv_label in LABEL_TABLE}

# Image counts per (class, domain) established by the audit. A different dataset must fail loudly.
EXPECTED_COUNTS = {
    ("Brown Spot", "white"): 90,
    ("Leaf Scald", "white"): 143,
    ("Rice Blast", "white"): 198,
    ("Rice Tungro", "white"): 119,
    ("Sheath Blight", "white"): 219,
    ("Brown Spot", "field"): 49,
    ("Leaf Scald", "field"): 74,
    ("Rice Blast", "field"): 74,
    ("Rice Tungro", "field"): 76,
    ("Sheath Blight", "field"): 64,
}
EXPECTED_TOTAL = sum(EXPECTED_COUNTS.values())  # 1106

CSV_FILENAME = "Dhan-Shomadhan_picture_Information.csv"


class DatasetError(Exception):
    """The dataset on disk is not what the code expects. The message names the offending path."""


def warn(message: str) -> None:
    print(f"warning: {message}", file=sys.stderr)


def extended_path(absolute_path: str, windows: bool) -> str:
    """Windows silently strips trailing spaces from path components (the dataset has folders called
    'Field Background  '). The extended-length prefix switches that off. No-op elsewhere."""
    if not windows or absolute_path.startswith("\\\\?\\"):
        return absolute_path
    if absolute_path.startswith("\\\\"):  # UNC share: \\server\share -> \\?\UNC\server\share
        return "\\\\?\\UNC\\" + absolute_path[2:]
    return "\\\\?\\" + absolute_path


def fs_path(path) -> str:
    """The string to hand to open()/os functions for ``path`` (absolute, Windows-safe)."""
    return extended_path(os.path.abspath(os.fspath(path)), os.name == "nt")


def read_file_bytes(path) -> bytes:
    with open(fs_path(path), "rb") as handle:
        return handle.read()


def write_bytes_atomic(path: Path, data: bytes) -> None:
    """Write via a temp file and rename, so an interrupted run never leaves a truncated file."""
    path = Path(path)
    os.makedirs(fs_path(path.parent), exist_ok=True)
    tmp = path.with_name(path.name + f".tmp{os.getpid()}")
    with open(fs_path(tmp), "wb") as handle:
        handle.write(data)
    os.replace(fs_path(tmp), fs_path(path))


def write_text_atomic(path: Path, text: str) -> None:
    write_bytes_atomic(path, text.encode("utf-8"))  # bytes: identical files on every OS (no CRLF)


def find_images(data_root: Path) -> list[Path]:
    """All JPEGs under data_root (recursive, case-insensitive), skipping the generated _work/ folder."""
    found = []
    for dirpath, dirnames, filenames in Path(data_root).walk():
        if dirpath == Path(data_root) and WORK_DIRNAME in dirnames:
            dirnames.remove(WORK_DIRNAME)
        for filename in filenames:
            if Path(filename).suffix.lower() in IMAGE_SUFFIXES:
                found.append(dirpath / filename)
    return sorted(found)


def classify_filename(filename: str) -> tuple[str, str]:
    """(class, domain) from the filename prefix; longest prefix wins. Raises DatasetError."""
    lowered = filename.lower()
    for prefix, cls, domain, _ in PREFIXES_LONGEST_FIRST:
        if lowered.startswith(prefix):
            return cls, domain
    known = ", ".join(row[0] for row in LABEL_TABLE)
    raise DatasetError(f"{filename}: filename matches no known prefix ({known})")

## Reading the EXIF orientation tag without PIL

381 of the 1106 images carry an EXIF rotation tag. The tag is stored in the index and is also what
`read_rgb` (below) uses to rotate the pixels, so it has to be parsed from the JPEG bytes directly.

In [ ]:
#| export
def _tiff_orientation(tiff: bytes) -> int:
    """Orientation (1..8) from the TIFF block of an EXIF segment; 1 if absent."""
    if len(tiff) < 8 or tiff[:2] not in (b"II", b"MM"):
        return 1
    endian = "<" if tiff[:2] == b"II" else ">"
    if struct.unpack_from(endian + "H", tiff, 2)[0] != 42:
        return 1
    offset = struct.unpack_from(endian + "I", tiff, 4)[0]
    if offset + 2 > len(tiff):
        return 1
    (count,) = struct.unpack_from(endian + "H", tiff, offset)
    for entry in range(count):
        base = offset + 2 + 12 * entry
        if base + 12 > len(tiff):
            break
        tag, kind, n_values = struct.unpack_from(endian + "HHI", tiff, base)
        if tag == 0x0112 and kind == 3 and n_values == 1:  # SHORT, one value, stored inline
            value = struct.unpack_from(endian + "H", tiff, base + 8)[0]
            return value if 1 <= value <= 8 else 1
    return 1


def parse_exif_orientation(data: bytes) -> int:
    """EXIF orientation (1..8) of a JPEG given its bytes. 1 means upright (also when there is no tag)."""
    if data[:2] != b"\xff\xd8":
        return 1
    pos, end = 2, len(data)
    while pos + 4 <= end:
        if data[pos] != 0xFF:
            break
        marker = data[pos + 1]
        if marker == 0xFF:  # fill byte
            pos += 1
            continue
        if marker == 0x01 or 0xD0 <= marker <= 0xD8:  # markers without a length
            pos += 2
            continue
        if marker in (0xD9, 0xDA):  # end of image / start of scan: no EXIF after this
            break
        (length,) = struct.unpack_from(">H", data, pos + 2)
        if marker == 0xE1 and data[pos + 4 : pos + 10] == b"Exif\x00\x00":
            return _tiff_orientation(data[pos + 10 : pos + 2 + length])
        pos += 2 + length
    return 1

## Dataset index

`build_index` walks the tree, labels every JPEG from its filename prefix, cross-checks the labels against
the CSV, checks the per class x domain counts against `EXPECTED_COUNTS` and returns the records.
`write_index` stores them in `_work/index.json`, which every later phase reads through `load_index`.

In [ ]:
#| export
@dataclass(frozen=True)
class Record:
    filename: str  # primary key (unique across the dataset)
    rel_path: str  # path relative to the data root, '/' separated
    cls: str
    cls_idx: int
    domain: str  # "white" or "field"
    exif_orientation: int  # 1..8; 1 = upright or no tag
    sha1: str  # of the file bytes; "" when hashing was skipped (smoke only)

    @property
    def stratum(self) -> tuple[str, str]:
        return (self.cls, self.domain)


def work_dir(data_root) -> Path:
    return Path(data_root) / WORK_DIRNAME


def index_path(data_root) -> Path:
    return work_dir(data_root) / "index.json"


def read_csv_labels(csv_path: Path) -> dict[str, str]:
    """{picture name: Diseases string}. The CSV has an unnamed index column, then `pictureName` and
    `Diseases`; it may start with a UTF-8 BOM."""
    with open(fs_path(csv_path), newline="", encoding="utf-8-sig") as handle:
        reader = csv.reader(handle)
        header = [cell.strip().lower() for cell in next(reader, [])]
        try:
            name_col, label_col = header.index("picturename"), header.index("diseases")
        except ValueError:
            raise DatasetError(f"{csv_path}: expected columns 'pictureName' and 'Diseases', found {header}") from None
        labels = {}
        for row in reader:
            if len(row) <= max(name_col, label_col):
                continue
            labels[row[name_col].strip()] = row[label_col].strip()
    return labels


def find_csv(data_root: Path) -> Path | None:
    path = Path(data_root) / CSV_FILENAME
    if path.is_file():
        return path
    others = sorted(Path(data_root).glob("*.csv"))
    return others[0] if len(others) == 1 else None


def check_against_csv(records: list[Record], csv_path: Path, *, require_complete: bool) -> None:
    csv_labels = read_csv_labels(csv_path)
    for record in records:
        if record.filename not in csv_labels:
            raise DatasetError(f"{record.rel_path}: not listed in {csv_path.name}")
        csv_string = csv_labels[record.filename]
        if csv_string not in CSV_LABELS:
            raise DatasetError(f"{csv_path.name}: unknown Diseases string {csv_string!r} for {record.filename}")
        if CSV_LABELS[csv_string] != (record.cls, record.domain):
            raise DatasetError(
                f"{record.rel_path}: filename prefix says {(record.cls, record.domain)} "
                f"but {csv_path.name} says {CSV_LABELS[csv_string]} ({csv_string!r})"
            )
    if require_complete:
        missing = sorted(set(csv_labels) - {r.filename for r in records})
        if missing:
            raise DatasetError(f"{csv_path.name} lists {len(missing)} images that are not on disk, e.g. {missing[:3]}")


def check_expected_counts(records: list[Record], expected: dict[tuple[str, str], int]) -> None:
    actual = Counter(r.stratum for r in records)
    problems = [
        f"{key[0]} / {key[1]}: expected {expected.get(key, 0)}, found {actual.get(key, 0)}"
        for key in sorted(set(expected) | set(actual))
        if expected.get(key, 0) != actual.get(key, 0)
    ]
    if len(records) != sum(expected.values()):
        problems.insert(0, f"total: expected {sum(expected.values())}, found {len(records)}")
    if problems:
        raise DatasetError("this is not the expected dataset (counts differ):\n  " + "\n  ".join(problems))


EXIF_HEADER_BYTES = 128 * 1024  # the EXIF segment is at the very start of the file


def build_index(
    data_root,
    *,
    expected_counts: dict[tuple[str, str], int] | None = EXPECTED_COUNTS,
    with_sha1: bool = True,
) -> list[Record]:
    """Index every JPEG under ``data_root`` (see the section text above), sorted by filename.

    ``expected_counts=None`` skips the count assertion and the CSV completeness check. That is only
    for tiny synthetic trees in tests and for the smoke check on a partial tree; every real run
    (`prepare`) uses the default, so a different dataset fails loudly.
    ``with_sha1=False`` reads only the file headers (fast); records then have ``sha1 == ""``.
    """
    data_root = Path(data_root)
    if not data_root.is_dir():
        raise DatasetError(f"data root {data_root} is not a directory (see --data-root)")
    records, seen = [], {}
    for path in find_images(data_root):
        cls, domain = classify_filename(path.name)
        if path.name in seen:
            raise DatasetError(f"duplicate filename {path.name}: {seen[path.name]} and {path}")
        seen[path.name] = path
        with open(fs_path(path), "rb") as handle:
            data = handle.read() if with_sha1 else handle.read(EXIF_HEADER_BYTES)
        records.append(
            Record(
                filename=path.name,
                rel_path=path.relative_to(data_root).as_posix(),
                cls=cls,
                cls_idx=CLASS_NAMES.index(cls),
                domain=domain,
                exif_orientation=parse_exif_orientation(data),
                sha1=hashlib.sha1(data).hexdigest() if with_sha1 else "",
            )
        )
    records.sort(key=lambda r: r.filename)
    csv_path = find_csv(data_root)
    if csv_path is None:
        warn(f"no label CSV found in {data_root}; labels come from filename prefixes only")
    else:
        check_against_csv(records, csv_path, require_complete=expected_counts is not None)
    if expected_counts is not None:
        check_expected_counts(records, expected_counts)
    return records


def write_json(path: Path, obj, *, indent: int | None = 1) -> None:
    write_text_atomic(path, json.dumps(obj, indent=indent, sort_keys=False) + "\n")


def write_index(data_root, records: list[Record]) -> Path:
    path = index_path(data_root)
    rows = ",\n".join("  " + json.dumps(asdict(r)) for r in records)  # one record per line: readable diffs
    write_text_atomic(path, '{\n "format": 1,\n "records": [\n' + rows + "\n ]\n}\n")
    return path


def load_index(data_root) -> list[Record]:
    """The records written by `prepare`, sorted by filename."""
    path = index_path(data_root)
    if not path.is_file():
        raise DatasetError(f"{path} does not exist; run the `prepare` command first")
    payload = json.loads(read_file_bytes(path).decode("utf-8"))
    if payload.get("format") != 1:
        raise DatasetError(f"{path}: unsupported index format {payload.get('format')!r}")
    return [Record(**row) for row in payload["records"]]

## EXIF-correct image IO

OpenCV 4.12 does apply the EXIF orientation by default (checked against `PIL.ImageOps.exif_transpose`
on real images with orientations 3, 6 and 8: identical pixels, also through `cv2.imdecode`). We do not
rely on that: decoding uses `IMREAD_IGNORE_ORIENTATION` and the orientation parsed above is applied
explicitly with numpy, so behaviour cannot change with the OpenCV build or flag defaults. After EXIF
every image is rotated to **portrait** (landscape images turn 90 degrees clockwise).

In [ ]:
#| export
def apply_orientation(rgb: np.ndarray, orientation: int) -> np.ndarray:
    """Undo an EXIF orientation tag (same conventions as PIL.ImageOps.exif_transpose)."""
    if orientation == 2:
        return rgb[:, ::-1]
    if orientation == 3:
        return np.rot90(rgb, 2)
    if orientation == 4:
        return rgb[::-1]
    if orientation == 5:  # transpose
        return rgb.transpose(1, 0, 2)
    if orientation == 6:  # stored rotated 90 degrees counter-clockwise: rotate clockwise
        return np.rot90(rgb, -1)
    if orientation == 7:  # transverse
        return np.rot90(rgb.transpose(1, 0, 2), 2)
    if orientation == 8:
        return np.rot90(rgb, 1)
    return rgb


def to_portrait(rgb: np.ndarray) -> tuple[np.ndarray, bool]:
    """Rotate a landscape image 90 degrees clockwise. Returns (image, was_rotated)."""
    if rgb.shape[1] > rgb.shape[0]:
        return np.rot90(rgb, -1), True
    return rgb, False


def decode_oriented(data: bytes, source: str = "<bytes>") -> np.ndarray:
    """JPEG bytes -> (H, W, 3) uint8 RGB with the EXIF orientation applied (not yet portrait)."""
    bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    if bgr is None:
        raise DatasetError(f"{source}: cannot decode image")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return apply_orientation(rgb, parse_exif_orientation(data))


def read_rgb(path) -> np.ndarray:
    """(H, W, 3) uint8 RGB, EXIF applied, always portrait (H >= W). Works for paths that OpenCV's own
    file reader struggles with (non-ASCII, Windows trailing-space folders) because we read the bytes."""
    rgb, _ = to_portrait(decode_oriented(read_file_bytes(path), str(path)))
    return np.ascontiguousarray(rgb)

## dHash

A 64-bit **difference hash**: grayscale, resize to 9x8 with `INTER_AREA`, one bit per pair of
horizontally adjacent pixels (left < right). Used by the near-duplicate audit below; it is computed
while caching so that the audit never has to decode the images again.

In [ ]:
#| export
DHASH_SIZE = (9, 8)  # width, height of the resized gray image -> 8 x 8 = 64 bits


def dhash_bits(rgb: np.ndarray) -> np.ndarray:
    """64 bits (uint8 0/1) of the difference hash of an RGB image."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, DHASH_SIZE, interpolation=cv2.INTER_AREA).astype(np.int16)
    return (small[:, 1:] > small[:, :-1]).ravel().astype(np.uint8)


def dhash_hex(bits: np.ndarray) -> str:
    return np.packbits(bits).tobytes().hex()


def hex_to_bits(text: str) -> np.ndarray:
    return np.unpackbits(np.frombuffer(bytes.fromhex(text), dtype=np.uint8))

## Resolution cache

Decoding an 8 MP JPEG every epoch would dominate training time. Every image is therefore decoded
once (EXIF applied, portrait), shrunk so that its **long side is 1280 px** (aspect kept, never
upscaled, `INTER_AREA`) and stored as a JPEG with quality 95 in `_work/cache/<filename>`. 1280 px leaves
headroom for 448 px inputs plus random-resized-crop zoom and for Method B crops.

`_work/cache/manifest.json` remembers, per filename, the sha1 of the source file the cached copy was
built from, the cached size and the dHash of the cached-resolution image (used by the near-duplicate
audit). A cached image is reused only if the file exists and that sha1 equals the one in the index, so
the step is idempotent and notices a changed dataset.

Work is spread over a `ProcessPoolExecutor`. The worker is a top-level function so that it can be
pickled under the `spawn` start method (macOS, Windows); the pool is only used when worker processes
can re-import this module (`python <file>.py ...`, or a `fork` platform).

In [ ]:
#| export
CACHE_LONG_SIDE = 1280
CACHE_JPEG_QUALITY = 95
MAX_WORKERS = 8


def cache_dir(data_root) -> Path:
    return work_dir(data_root) / "cache"


def manifest_path(data_root) -> Path:
    return cache_dir(data_root) / "manifest.json"


def default_workers() -> int:
    try:
        available = len(os.sched_getaffinity(0))
    except AttributeError:  # macOS, Windows
        available = os.cpu_count() or 1
    return max(1, min(available, MAX_WORKERS))


def pool_is_safe() -> bool:
    """True if worker processes can import the functions we send them: always under `fork`; under
    `spawn` only when this file runs as the main script (children re-run it as `__mp_main__`)."""
    return multiprocessing.get_start_method() == "fork" or __name__ == "__main__"


def _pool_init() -> None:
    cv2.setNumThreads(1)  # one process per core already; avoid oversubscription


def run_parallel(func, tasks, *, workers: int, mp_context=None, chunksize: int = 4):
    """Yield ``func(task)`` for every task, in order. Uses a process pool when that is possible and
    worthwhile, otherwise runs inline. ``func`` must be a picklable top-level function."""
    tasks = list(tasks)
    use_pool = workers > 1 and len(tasks) > 1 and (mp_context is not None or pool_is_safe())
    if not use_pool:
        for task in tasks:
            yield func(task)
        return
    with ProcessPoolExecutor(
        max_workers=min(workers, len(tasks)), mp_context=mp_context, initializer=_pool_init
    ) as executor:
        yield from executor.map(func, tasks, chunksize=chunksize)


def resize_long_side(rgb: np.ndarray, long_side: int) -> np.ndarray:
    height, width = rgb.shape[:2]
    scale = long_side / max(height, width)
    if scale >= 1.0:
        return rgb
    size = (max(1, round(width * scale)), max(1, round(height * scale)))
    return cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)


def encode_jpeg(rgb: np.ndarray, quality: int) -> bytes:
    ok, buffer = cv2.imencode(".jpg", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise DatasetError("JPEG encoding failed")
    return buffer.tobytes()


def _cache_one(task: tuple) -> tuple[str, dict]:
    """Worker: source file -> cached portrait JPEG. Returns (filename, manifest entry without sha1)."""
    filename, source, target, long_side, quality = task
    try:
        rgb, rotated = to_portrait(decode_oriented(read_file_bytes(source), source))
        small = resize_long_side(np.ascontiguousarray(rgb), long_side)
        write_bytes_atomic(Path(target), encode_jpeg(small, quality))
    except Exception as exc:
        raise DatasetError(f"{source}: caching failed: {exc}") from exc
    return filename, {
        "width": int(small.shape[1]),
        "height": int(small.shape[0]),
        "rotated_landscape": bool(rotated),
        "dhash": dhash_hex(dhash_bits(small)),  # of the downsized image, before JPEG encoding
    }


@dataclass
class CacheStats:
    total: int
    built: int
    reused: int
    rotated_landscape: int  # images (of `total`) that were landscape after EXIF and were turned to portrait


def load_manifest(data_root) -> dict:
    path = manifest_path(data_root)
    if not path.is_file():
        return {}
    try:
        return json.loads(read_file_bytes(path).decode("utf-8"))
    except (OSError, ValueError):
        return {}  # a damaged manifest only means "rebuild everything"


def write_manifest(data_root, manifest: dict) -> None:
    write_json(manifest_path(data_root), dict(sorted(manifest.items())))


def build_cache(
    data_root,
    records: list[Record],
    *,
    workers: int | None = None,
    mp_context=None,
    long_side: int = CACHE_LONG_SIDE,
    quality: int = CACHE_JPEG_QUALITY,
    output_root=None,
    log=None,
) -> CacheStats:
    """Cache every record (see section text). Idempotent. ``records`` need their sha1 (not the
    ``with_sha1=False`` kind). Images are read from ``data_root``; the cache is written under
    ``<output_root>/_work/cache`` (default: the same data root; the smoke check uses a temp folder)."""
    if any(not r.sha1 for r in records):
        raise DatasetError("build_cache needs records with sha1 (build_index(..., with_sha1=True))")
    data_root = Path(data_root)
    output_root = Path(output_root) if output_root is not None else data_root
    root = cache_dir(output_root)
    manifest = load_manifest(output_root)

    def current(record: Record) -> bool:
        entry = manifest.get(record.filename)
        return (
            bool(entry)
            and entry.get("sha1") == record.sha1
            and "dhash" in entry
            and (root / record.filename).is_file()
        )

    todo = [r for r in records if not current(r)]
    tasks = [(r.filename, str(data_root / r.rel_path), str(root / r.filename), long_side, quality) for r in todo]
    sha_of = {r.filename: r.sha1 for r in todo}
    done = 0
    try:
        for filename, entry in run_parallel(_cache_one, tasks, workers=workers or default_workers()):
            manifest[filename] = {"sha1": sha_of[filename], **entry}
            done += 1
            if log and done % 200 == 0:
                log(f"cache: {done}/{len(tasks)}")
    finally:
        if done:
            write_manifest(output_root, manifest)  # also after an interrupted run: finished images are kept
    rotated = sum(bool(manifest[r.filename]["rotated_landscape"]) for r in records)
    return CacheStats(total=len(records), built=len(todo), reused=len(records) - len(todo), rotated_landscape=rotated)


def read_cached_rgb(data_root, filename: str) -> np.ndarray:
    """A cached image: (H, W, 3) uint8 RGB, portrait, long side <= 1280."""
    return read_rgb(cache_dir(data_root) / filename)

## Near-duplicate audit (leakage guard)

If the same leaf appears twice and lands in train and test, test accuracy is inflated; small
agricultural datasets often contain bursts of near-identical shots. Every pair of images whose dHash
Hamming distance is <= 6 is a **candidate**; all candidates, within and across domains, go into
`_work/results/near_duplicates.csv`.

The dHash is a very coarse fingerprint. On this dataset a white-background image is a thin leaf on a
plain wall, so unrelated leaves hash alike and hundreds of candidates are not duplicates at all. Each
candidate therefore also gets a second, independent score: the number of RANSAC inliers when SIFT
keypoints of the two images are matched (`feature_inliers`; a genuine duplicate shares hundreds,
unrelated leaves a handful). The scores only **rank** the candidates for review (CSV order and
`pair_id` = review order). The code never decides what a duplicate is: a person looks at the contact
sheets `_work/figures/near_duplicates_<n>.png` (the top-ranked pairs) and writes the confirmed groups to
`_work/duplicate_groups.json`, a list of lists of filenames. The splits keep every group in one split. Until that file
exists the splits treat every image as its own unit and `prepare` warns.

In [ ]:
#| export
NEAR_DUPLICATE_MAX_DISTANCE = 6
DUPLICATE_GROUPS_FILENAME = "duplicate_groups.json"
FEATURE_LONG_SIDE = 1000  # images are downscaled to this long side before SIFT
FEATURE_MAX_KEYPOINTS = 3000
FEATURE_CONTRAST = 0.01  # below OpenCV's 0.04 default: leaves on a plain wall have few strong corners
FEATURE_RATIO = 0.75  # Lowe ratio test
FEATURE_RANSAC_PX = 6.0
SHEET_MAX_PAIRS = 48  # pairs shown on contact sheets (the top of the ranking)
SHEET_PAIRS_PER_PAGE = 8


def hamming_matrix(bits: np.ndarray) -> np.ndarray:
    """All pairwise Hamming distances of an (N, 64) 0/1 matrix, as an (N, N) int matrix."""
    b = bits.astype(np.float32)
    return np.rint(b @ (1.0 - b).T + (1.0 - b) @ b.T).astype(np.int32)


def find_near_duplicates(bits: np.ndarray, max_distance: int = NEAR_DUPLICATE_MAX_DISTANCE) -> list[tuple[int, int, int]]:
    """(i, j, distance) with i < j and distance <= max_distance, sorted by distance then index."""
    dist = hamming_matrix(bits)
    rows, cols = np.nonzero(np.triu(dist <= max_distance, k=1))
    pairs = [(int(i), int(j), int(dist[i, j])) for i, j in zip(rows, cols)]
    return sorted(pairs, key=lambda p: (p[2], p[0], p[1]))


def sift_features(rgb: np.ndarray):
    """(keypoint coordinates (K, 2) float32, descriptors (K, 128) float32) at FEATURE_LONG_SIDE."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    scale = FEATURE_LONG_SIDE / max(gray.shape)
    if scale < 1.0:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    keypoints, descriptors = cv2.SIFT_create(nfeatures=FEATURE_MAX_KEYPOINTS, contrastThreshold=FEATURE_CONTRAST).detectAndCompute(gray, None)
    points = np.float32([k.pt for k in keypoints]).reshape(-1, 2)
    return points, descriptors


def feature_inliers(features_a, features_b) -> int:
    """Number of ratio-test SIFT matches consistent with one similarity transform (RANSAC)."""
    (points_a, desc_a), (points_b, desc_b) = features_a, features_b
    if desc_a is None or desc_b is None or len(desc_a) < 8 or len(desc_b) < 8:
        return 0
    matches = cv2.BFMatcher(cv2.NORM_L2).knnMatch(desc_a, desc_b, k=2)
    good = [m[0] for m in matches if len(m) == 2 and m[0].distance < FEATURE_RATIO * m[1].distance]
    if len(good) < 8:
        return 0
    src = points_a[[m.queryIdx for m in good]]
    dst = points_b[[m.trainIdx for m in good]]
    _, mask = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=FEATURE_RANSAC_PX)
    return int(mask.sum()) if mask is not None else 0


@dataclass(frozen=True)
class DuplicatePair:
    pair_id: int  # review rank, 1 = most likely a duplicate
    hamming: int
    feature_inliers: int
    a: Record
    b: Record

    @property
    def scope(self) -> str:
        return "within domain" if self.a.domain == self.b.domain else "across domains"


def load_dhashes(data_root, records: list[Record]) -> np.ndarray:
    """(N, 64) uint8 matrix of the dHashes stored in the cache manifest (run build_cache first)."""
    manifest = load_manifest(data_root)
    missing = [r.filename for r in records if "dhash" not in manifest.get(r.filename, {})]
    if missing:
        raise DatasetError(f"no dHash for {len(missing)} images (e.g. {missing[:3]}); run `prepare` to build the cache")
    return np.stack([hex_to_bits(manifest[r.filename]["dhash"]) for r in records]) if records else np.zeros((0, 64), np.uint8)


def duplicate_pairs(
    data_root, records: list[Record], bits: np.ndarray, max_distance: int = NEAR_DUPLICATE_MAX_DISTANCE
) -> list[DuplicatePair]:
    """All candidate pairs, ranked: most feature inliers first, then smallest Hamming distance, then names."""
    candidates = find_near_duplicates(bits, max_distance)
    needed = sorted({i for i, j, _ in candidates} | {j for i, j, _ in candidates})
    features = {i: sift_features(read_cached_rgb(data_root, records[i].filename)) for i in needed}
    scored = [
        (-feature_inliers(features[i], features[j]), d, records[i].filename, records[j].filename, records[i], records[j])
        for i, j, d in candidates
    ]
    scored.sort(key=lambda t: t[:4])
    return [DuplicatePair(rank, d, -neg, a, b) for rank, (neg, d, _, _, a, b) in enumerate(scored, start=1)]


def duplicate_stats(pairs: list[DuplicatePair]) -> dict:
    """Counts of candidate pairs per Hamming distance and per scope (within / across domains)."""
    by_distance = Counter(p.hamming for p in pairs)
    by_scope = Counter(p.scope for p in pairs)
    inliers = [p.feature_inliers for p in pairs]
    return {
        "total": len(pairs),
        "by_distance": {d: by_distance[d] for d in sorted(by_distance)},
        "within_domain": by_scope["within domain"],
        "across_domains": by_scope["across domains"],
        "same_class": sum(p.a.cls == p.b.cls for p in pairs),
        "images_involved": len({p.a.filename for p in pairs} | {p.b.filename for p in pairs}),
        "max_feature_inliers": max(inliers, default=0),
    }


def results_dir(data_root) -> Path:
    return work_dir(data_root) / "results"


def figures_dir(data_root) -> Path:
    return work_dir(data_root) / "figures"


def write_duplicate_report(data_root, pairs: list[DuplicatePair]) -> Path:
    """`_work/results/near_duplicates.csv`: every candidate pair, in review order."""
    path = results_dir(data_root) / "near_duplicates.csv"
    lines = ["pair_id,hamming,feature_inliers,scope,file_a,file_b,class_a,class_b,domain_a,domain_b"]
    for p in pairs:
        lines.append(
            f"{p.pair_id},{p.hamming},{p.feature_inliers},{p.scope},{p.a.filename},{p.b.filename},"
            f"{p.a.cls},{p.b.cls},{p.a.domain},{p.b.domain}"
        )
    write_text_atomic(path, "\n".join(lines) + "\n")
    return path


def merge_groups(groups: list[list[str]]) -> list[list[str]]:
    """Transitive merge: groups that share a filename become one group. Sorted, deterministic."""
    parent: dict[str, str] = {}

    def find(x: str) -> str:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for group in groups:
        for name in group:
            parent.setdefault(name, name)
        for name in group[1:]:
            parent[find(name)] = find(group[0])
    merged: dict[str, list[str]] = {}
    for name in parent:
        merged.setdefault(find(name), []).append(name)
    return sorted((sorted(members) for members in merged.values() if len(members) > 1), key=lambda m: m[0])


def duplicate_groups_path(data_root) -> Path:
    return work_dir(data_root) / DUPLICATE_GROUPS_FILENAME


def load_duplicate_groups(data_root, filenames) -> list[list[str]] | None:
    """The human-confirmed duplicate groups, transitively merged; ``None`` if the file does not exist
    (duplicates are then UNCONFIRMED, which is different from an empty list: "confirmed, none")."""
    path = duplicate_groups_path(data_root)
    if not path.is_file():
        return None
    try:
        raw = json.loads(read_file_bytes(path).decode("utf-8"))
    except ValueError as exc:
        raise DatasetError(f"{path}: not valid JSON: {exc}") from exc
    if not isinstance(raw, list) or not all(isinstance(g, list) and all(isinstance(n, str) for n in g) for g in raw):
        raise DatasetError(f"{path}: expected a list of lists of filenames")
    known = set(filenames)
    unknown = sorted({n for g in raw for n in g} - known)
    if unknown:
        raise DatasetError(f"{path}: {len(unknown)} filenames are not in the index, e.g. {unknown[:5]}")
    return merge_groups(raw)

## Plotting helpers and the duplicate contact sheets

Figures use the non-interactive Agg backend (imported lazily, so worker processes stay light) and
the colour-blind-safe Okabe-Ito palette.

In [ ]:
#| export
# Okabe-Ito colour-blind-safe palette.
OKABE_ITO = {
    "black": "#000000",
    "orange": "#E69F00",
    "sky": "#56B4E9",
    "green": "#009E73",
    "yellow": "#F0E442",
    "blue": "#0072B2",
    "vermilion": "#D55E00",
    "purple": "#CC79A7",
}
# One colour per domain (2 series: they must stay distinguishable in grayscale print as well).
DOMAIN_COLORS = {"white": OKABE_ITO["blue"], "field": OKABE_ITO["orange"]}
FIGURE_DPI = 300


def pyplot():
    """matplotlib.pyplot on the Agg backend (no display needed, also on Colab and on servers)."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    return plt


def thumbnail(rgb: np.ndarray, height: int) -> np.ndarray:
    scale = height / rgb.shape[0]
    return cv2.resize(rgb, (max(1, round(rgb.shape[1] * scale)), height), interpolation=cv2.INTER_AREA)


def save_duplicate_sheets(
    data_root,
    pairs: list[DuplicatePair],
    *,
    max_pairs: int = SHEET_MAX_PAIRS,
    per_page: int = SHEET_PAIRS_PER_PAGE,
) -> list[Path]:
    """Contact sheets `near_duplicates_<n>.png` of the top-ranked pairs, side by side, labelled with
    filename, class, domain, dHash distance and feature inliers. Old pages are removed first."""
    plt = pyplot()
    out_dir = figures_dir(data_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob("near_duplicates*.png"):
        old.unlink()
    shown = pairs[:max_pairs]
    cols, rows = 4, max(1, per_page // 4)
    pages = [shown[i : i + per_page] for i in range(0, len(shown), per_page)]
    paths = []
    for number, chunk in enumerate(pages, start=1):
        fig = plt.figure(figsize=(20, 11.4 if rows == 2 else 6), dpi=100)
        grid = fig.add_gridspec(rows, cols, left=0.005, right=0.995, top=0.935, bottom=0.005, wspace=0.05, hspace=0.1)
        for k, pair in enumerate(chunk):
            inner = grid[k // cols, k % cols].subgridspec(2, 2, height_ratios=[0.055, 1], wspace=0.03, hspace=0.01)
            head = fig.add_subplot(inner[0, :])
            head.axis("off")
            head.text(
                0.5, 0.5,
                f"#{pair.pair_id}   dHash {pair.hamming}   inliers {pair.feature_inliers}   ({pair.scope})",
                ha="center", va="center", fontsize=11, fontweight="bold",
            )
            for side, record in enumerate((pair.a, pair.b)):
                ax = fig.add_subplot(inner[1, side])
                ax.imshow(thumbnail(read_cached_rgb(data_root, record.filename), 560))
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_edgecolor(DOMAIN_COLORS[record.domain])
                    spine.set_linewidth(2.5)
                ax.set_xlabel(f"{record.filename}\n{record.cls}, {record.domain}", fontsize=9)
        first, last = chunk[0].pair_id, chunk[-1].pair_id
        fig.suptitle(
            f"Near-duplicate candidates, page {number}/{len(pages)} (pairs #{first}-#{last} of {len(pairs)}; "
            "border colour: blue = white background, orange = field)",
            fontsize=13, y=0.985,
        )
        path = out_dir / f"near_duplicates_{number:02d}.png"
        fig.savefig(path, dpi=100)
        plt.close(fig)
        paths.append(path)
    return paths

## Stratified, group-aware splits

Five splits (seeds 0-4), 70 / 15 / 15 train / val / test, stratified by **class x domain**, so every
split has about 50 field test images. Duplicate groups (see above) are kept in one split.

Algorithm for seed `k`, applied to each class x domain stratum (strata visited in `CLASS_NAMES` x
`DOMAINS` order):

1. The strata's **units** are single images, or confirmed duplicate groups counted as one unit. A
   group whose members span several strata belongs to the stratum of its **first member by filename**
   (cross-domain groups are expected to be rare; the choice only needs to be deterministic).
2. Sort the units by their member filenames, then permute them with a fresh `np.random.default_rng(k)`.
3. With `n` = number of units, the first `round(0.15 n)` go to test, the next `round(0.15 n)` to val,
   the rest to train. `round` is half-up integer arithmetic (`(15 n + 50) // 100`), not Python's
   banker's rounding on a float, so the result cannot differ between platforms.

`sha256` identifies a split: it is the SHA-256 of the UTF-8 bytes of
`json.dumps({"seed": k, "train": [...], "val": [...], "test": [...]}, sort_keys=True, separators=(",", ":"))`
where each list holds the sorted filenames. Every run's `metrics.json` records it, and `summarize`
refuses to aggregate runs whose sha differs from the committed split file.

In [ ]:
#| export
SPLIT_SEEDS = (0, 1, 2, 3, 4)
SPLIT_PERCENT = 15  # test and val each; train gets the remaining 70


def split_size(n_units: int, percent: int = SPLIT_PERCENT) -> int:
    """round(percent / 100 * n) with halves rounded up, in integer arithmetic."""
    return (percent * n_units + 50) // 100


def split_sha256(seed: int, train: list[str], val: list[str], test: list[str]) -> str:
    payload = json.dumps(
        {"seed": seed, "train": sorted(train), "val": sorted(val), "test": sorted(test)},
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def stratum_order(stratum: tuple[str, str]) -> tuple[int, int]:
    return (CLASS_NAMES.index(stratum[0]), DOMAINS.index(stratum[1]))


def build_units(records: list[Record], groups: list[list[str]] | None) -> dict[tuple[str, str], list[tuple[str, ...]]]:
    """{stratum: units}; a unit is a sorted tuple of filenames. ``groups=None`` (unconfirmed) and
    ``groups=[]`` (confirmed, none) both give one unit per image."""
    by_name = {r.filename: r for r in records}
    units: dict[tuple[str, str], list[tuple[str, ...]]] = {}
    grouped: set[str] = set()
    for group in merge_groups(groups or []):
        members = tuple(sorted(group))
        grouped.update(members)
        units.setdefault(by_name[members[0]].stratum, []).append(members)
    for record in records:
        if record.filename not in grouped:
            units.setdefault(record.stratum, []).append((record.filename,))
    return units


def make_split(records: list[Record], groups: list[list[str]] | None, seed: int) -> dict:
    """One split as a dict with keys seed, train, val, test (sorted filename lists) and sha256."""
    train: list[str] = []
    val: list[str] = []
    test: list[str] = []
    units_by_stratum = build_units(records, groups)
    for stratum in sorted(units_by_stratum, key=stratum_order):
        units = sorted(units_by_stratum[stratum])
        shuffled = [units[i] for i in np.random.default_rng(seed).permutation(len(units))]
        n_test = split_size(len(units))
        n_val = split_size(len(units))
        for target, chosen in (
            (test, shuffled[:n_test]),
            (val, shuffled[n_test : n_test + n_val]),
            (train, shuffled[n_test + n_val :]),
        ):
            target.extend(name for unit in chosen for name in unit)
    train, val, test = sorted(train), sorted(val), sorted(test)
    return {"seed": seed, "train": train, "val": val, "test": test, "sha256": split_sha256(seed, train, val, test)}


def splits_dir(data_root) -> Path:
    return work_dir(data_root) / "splits"


def split_path(data_root, seed: int) -> Path:
    return splits_dir(data_root) / f"split_{seed}.json"


def write_splits(
    data_root, records: list[Record], groups: list[list[str]] | None, seeds=SPLIT_SEEDS
) -> dict[int, str]:
    """Generate and write split_<k>.json. Returns {seed: "new" | "unchanged" | "CHANGED"} so that the
    caller can say loudly when an existing (committed) split differs from the regenerated one."""
    status = {}
    for seed in seeds:
        split = make_split(records, groups, seed)
        path = split_path(data_root, seed)
        before = None
        if path.is_file():
            try:
                before = json.loads(read_file_bytes(path).decode("utf-8")).get("sha256")
            except ValueError:
                before = ""
        status[seed] = "new" if before is None else "unchanged" if before == split["sha256"] else "CHANGED"
        if status[seed] != "unchanged":
            write_json(path, split)
    return status


def load_split(data_root, seed: int) -> dict:
    """A split written by `prepare`, with its sha256 re-computed and verified."""
    path = split_path(data_root, seed)
    if not path.is_file():
        raise DatasetError(f"{path} does not exist; run the `prepare` command first")
    split = json.loads(read_file_bytes(path).decode("utf-8"))
    expected = split_sha256(split["seed"], split["train"], split["val"], split["test"])
    if split.get("sha256") != expected:
        raise DatasetError(f"{path}: sha256 does not match its content (file edited by hand?)")
    return split


def split_counts(split: dict, records: list[Record]) -> dict:
    """{'train': {'white': n, 'field': n, 'all': n}, 'val': ..., 'test': ...}"""
    domain_of = {r.filename: r.domain for r in records}
    return {
        part: {**{d: sum(domain_of[n] == d for n in split[part]) for d in DOMAINS}, "all": len(split[part])}
        for part in ("train", "val", "test")
    }

## Dataset figures and tables for the report

* `_work/results/dataset_counts.md` - class x domain table with totals.
* `_work/figures/dataset_grid.png` - Figure 1: one sample per class and domain (fixed seed, EXIF-correct,
  taken from the cache).
* `_work/figures/class_domain_counts.png` - grouped bar chart.

Figures are 300 dpi, sized for a half-page-wide column (about 5.5 in wide, 8-9 pt text), use the
Okabe-Ito palette and always list the classes in `CLASS_NAMES` order.

In [ ]:
#| export
GRID_SEED = 42
DOMAIN_LABELS = {"white": "White background", "field": "Field background"}


def class_domain_counts(records: list[Record]) -> dict[tuple[str, str], int]:
    counts = Counter(r.stratum for r in records)
    return {(c, d): counts.get((c, d), 0) for c in CLASS_NAMES for d in DOMAINS}


def dataset_counts_markdown(records: list[Record]) -> str:
    counts = class_domain_counts(records)
    lines = ["| Class | White background | Field background | Total |", "|---|---:|---:|---:|"]
    for cls in CLASS_NAMES:
        white, field = counts[(cls, "white")], counts[(cls, "field")]
        lines.append(f"| {cls} | {white} | {field} | {white + field} |")
    white_total = sum(counts[(c, "white")] for c in CLASS_NAMES)
    field_total = sum(counts[(c, "field")] for c in CLASS_NAMES)
    lines.append(f"| **Total** | **{white_total}** | **{field_total}** | **{white_total + field_total}** |")
    return "\n".join(lines) + "\n"


def write_dataset_counts(data_root, records: list[Record]) -> Path:
    path = results_dir(data_root) / "dataset_counts.md"
    write_text_atomic(path, dataset_counts_markdown(records))
    return path


def save_dataset_grid(data_root, records: list[Record], path: Path | None = None, seed: int = GRID_SEED) -> Path:
    """2 rows (white, field) x 5 classes, one random (seeded) sample each, read from the cache."""
    plt = pyplot()
    path = path or figures_dir(data_root) / "dataset_grid.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    chosen = {}
    for domain in DOMAINS:
        for cls in CLASS_NAMES:
            names = sorted(r.filename for r in records if r.stratum == (cls, domain))
            chosen[(cls, domain)] = names[int(rng.integers(len(names)))] if names else None
    with plt.rc_context({"font.size": 8.5}):
        fig = plt.figure(figsize=(5.6, 4.72))
        grid = fig.add_gridspec(2, 5, left=0.065, right=0.995, top=0.94, bottom=0.004, wspace=0.05, hspace=0.02)
        for row, domain in enumerate(DOMAINS):
            for col, cls in enumerate(CLASS_NAMES):
                ax = fig.add_subplot(grid[row, col])
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
                name = chosen[(cls, domain)]
                if name is not None:
                    ax.imshow(thumbnail(read_cached_rgb(data_root, name), 760))
                if row == 0:
                    ax.set_title(cls, fontsize=8.5, fontweight="bold", pad=3)
                if col == 0:
                    ax.set_ylabel(DOMAIN_LABELS[domain], fontsize=8.5, fontweight="bold", labelpad=4)
        fig.savefig(path, dpi=FIGURE_DPI)
        plt.close(fig)
    return path


def save_class_domain_counts(data_root, records: list[Record], path: Path | None = None) -> Path:
    """Grouped bar chart: images per class, one bar per domain, value labels on the bars."""
    plt = pyplot()
    path = path or figures_dir(data_root) / "class_domain_counts.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    counts = class_domain_counts(records)
    x = np.arange(len(CLASS_NAMES))
    width = 0.38
    with plt.rc_context({"font.size": 9}):
        fig, ax = plt.subplots(figsize=(5.4, 3.3))
        for k, domain in enumerate(DOMAINS):
            values = [counts[(c, domain)] for c in CLASS_NAMES]
            bars = ax.bar(
                x + (k - 0.5) * width, values, width * 0.94, color=DOMAIN_COLORS[domain], label=DOMAIN_LABELS[domain]
            )
            ax.bar_label(bars, fontsize=8, padding=2)
        ax.set_xticks(x, [c.replace(" ", "\n") for c in CLASS_NAMES])
        ax.set_ylabel("Number of images")
        ax.set_ylim(0, max(counts.values(), default=1) * 1.15)
        ax.yaxis.grid(True, color="#DDDDDD", linewidth=0.6)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.legend(frameon=False, loc="upper left", fontsize=8.5)
        fig.tight_layout(pad=0.6)
        fig.savefig(path, dpi=FIGURE_DPI)
        plt.close(fig)
    return path

## `prepare` and the data smoke check

`prepare_data` runs T2.1-T2.6 in order and prints a one-screen summary. It is idempotent: the cache
skips images that are up to date, and the CSV, splits, tables and figures are regenerated
deterministically (the splits report whether a file on disk was `new`, `unchanged` or `CHANGED`).
`run_data_smoke` is the cheap version used by the `smoke` command: it never touches the real
`_work/` folder.

In [ ]:
#| export
def exif_summary(records: list[Record]) -> str:
    tagged = Counter(r.exif_orientation for r in records if r.exif_orientation != 1)
    detail = ", ".join(f"orientation {o}: {n}" for o, n in sorted(tagged.items()))
    return f"{sum(tagged.values())} images carry an EXIF rotation tag ({detail or 'none'}); all applied"


def prepare_data(data_root, *, expected_counts=EXPECTED_COUNTS, workers: int | None = None, out=print) -> dict:
    """index -> cache -> near-duplicate audit -> splits -> tables and figures. Returns a summary dict
    (also printed through ``out``). Raises DatasetError on any dataset problem."""
    import time

    start = time.time()
    data_root = Path(data_root)
    out(f"prepare: data root {data_root.resolve()}")
    records = build_index(data_root, expected_counts=expected_counts)
    write_index(data_root, records)
    counts = Counter(r.domain for r in records)
    out(f"  index       {len(records)} images: white {counts['white']}, field {counts['field']}; {len(CLASS_NAMES)} classes")
    out(f"  exif        {exif_summary(records)}")

    t = time.time()
    cache = build_cache(data_root, records, workers=workers, log=out)
    out(
        f"  portrait    {cache.rotated_landscape} images were landscape after EXIF and were turned 90 degrees clockwise"
    )
    out(
        f"  cache       {cache.total} images, long side {CACHE_LONG_SIDE} px, JPEG q{CACHE_JPEG_QUALITY}: "
        f"built {cache.built}, already up to date {cache.reused} ({time.time() - t:.0f} s)"
    )

    t = time.time()
    pairs = duplicate_pairs(data_root, records, load_dhashes(data_root, records))
    stats = duplicate_stats(pairs)
    write_duplicate_report(data_root, pairs)
    sheets = save_duplicate_sheets(data_root, pairs)
    by_distance = " ".join(f"{d}:{n}" for d, n in stats["by_distance"].items())
    out(
        f"  duplicates  {stats['total']} dHash candidate pairs (distance <= {NEAR_DUPLICATE_MAX_DISTANCE}): "
        f"{stats['within_domain']} within domain, {stats['across_domains']} across; by distance {by_distance}"
    )
    out(
        f"              best feature-match score {stats['max_feature_inliers']} (a real duplicate shares hundreds); "
        f"{len(sheets)} contact sheets in {figures_dir(data_root)} ({time.time() - t:.0f} s)"
    )
    groups = load_duplicate_groups(data_root, [r.filename for r in records])
    if groups is None:
        out(f"  WARNING     duplicates are UNCONFIRMED: no {duplicate_groups_path(data_root).name} in {work_dir(data_root)}.")
        out("              Every image is treated as its own unit in the splits. Review the contact sheets and the")
        out("              CSV, then write the confirmed groups (list of lists of filenames) to that file and re-run.")
    else:
        out(f"  duplicates  {len(groups)} confirmed groups ({sum(len(g) for g in groups)} images) are kept together in the splits")

    status = write_splits(data_root, records, groups)
    first = load_split(data_root, SPLIT_SEEDS[0])
    sizes = split_counts(first, records)
    out(
        f"  splits      seeds {SPLIT_SEEDS[0]}-{SPLIT_SEEDS[-1]}: train/val/test = "
        f"{sizes['train']['all']}/{sizes['val']['all']}/{sizes['test']['all']} "
        f"(test: {sizes['test']['white']} white, {sizes['test']['field']} field); "
        + ", ".join(f"{seed}: {state}" for seed, state in status.items())
    )
    if "CHANGED" in status.values():
        out("  WARNING     an existing split file differs from the regenerated one and was overwritten")
        out("              (expected after the duplicate groups changed; otherwise investigate before training)")
    out("              sha256 " + " ".join(load_split(data_root, s)["sha256"][:10] for s in SPLIT_SEEDS))

    write_dataset_counts(data_root, records)
    save_dataset_grid(data_root, records)
    save_class_domain_counts(data_root, records)
    out(f"  report      dataset_counts.md, dataset_grid.png, class_domain_counts.png in {work_dir(data_root)}")
    elapsed = time.time() - start
    out(f"prepare: done in {elapsed:.0f} s")
    return {
        "records": len(records),
        "cache": cache,
        "duplicates": stats,
        "duplicate_groups": groups,
        "splits": status,
        "seconds": elapsed,
    }


def synthetic_records(per_stratum: int = 20) -> list[Record]:
    """Fake index entries (no files) covering every class x domain stratum; for smoke checks and tests."""
    prefix = {(cls, domain): p for p, cls, domain, _ in LABEL_TABLE}
    return [
        Record(f"{prefix[(cls, domain)]}{i}.jpg", f"x/{prefix[(cls, domain)]}{i}.jpg", cls, CLASS_NAMES.index(cls), domain, 1, "")
        for cls in CLASS_NAMES
        for domain in DOMAINS
        for i in range(per_stratum)
    ]


def pick_smoke_records(records: list[Record], n: int = 20) -> list[Record]:
    """Up to ``n`` records: first one of each EXIF orientation 3/6/8 present, then round-robin over strata."""
    chosen: list[Record] = []
    for orientation in (3, 6, 8):
        match = next((r for r in records if r.exif_orientation == orientation), None)
        if match:
            chosen.append(match)
    queues = {}
    for r in records:
        queues.setdefault(r.stratum, []).append(r)
    order = sorted(queues, key=stratum_order)
    while len(chosen) < min(n, len(records)):
        progressed = False
        for stratum in order:
            while queues[stratum] and queues[stratum][0] in chosen:
                queues[stratum].pop(0)
            if queues[stratum] and len(chosen) < n:
                chosen.append(queues[stratum].pop(0))
                progressed = True
        if not progressed:
            break
    return chosen


def smoke_require(condition: bool, message: str) -> None:
    if not condition:
        raise DatasetError(f"data smoke check failed: {message}")


def run_data_smoke(data_root, n_images: int = 20, workers: int = 2, log=print) -> None:
    """Cheap end-to-end check of the data path on ~20 real images. Writes only to a temp folder."""
    data_root = Path(data_root)
    records = build_index(data_root, expected_counts=None, with_sha1=False)
    log(f"smoke: OK - index of {len(records)} images, labels match filename prefixes (and the CSV if present)")
    groups = load_duplicate_groups(data_root, [r.filename for r in records])
    log("smoke: OK - duplicate groups: " + ("not confirmed yet (no file)" if groups is None else f"{len(groups)} confirmed groups"))
    selected = [
        replace(r, sha1=hashlib.sha1(read_file_bytes(data_root / r.rel_path)).hexdigest())
        for r in pick_smoke_records(records, n_images)
    ]
    with tempfile.TemporaryDirectory(prefix="csci935-smoke-") as tmp:
        tmp_root = Path(tmp)
        stats = build_cache(data_root, selected, workers=workers, output_root=tmp_root)
        smoke_require(stats.built == len(selected), f"expected {len(selected)} cached images, built {stats.built}")
        for record in selected:
            rgb = read_cached_rgb(tmp_root, record.filename)
            smoke_require(rgb.ndim == 3 and rgb.shape[2] == 3 and rgb.dtype == np.uint8, f"{record.filename}: bad array")
            smoke_require(rgb.shape[0] >= rgb.shape[1], f"{record.filename}: cached image is not portrait")
            smoke_require(max(rgb.shape[:2]) <= CACHE_LONG_SIDE, f"{record.filename}: cached image too large")
        for record in [r for r in selected if r.exif_orientation != 1][:3]:
            full = read_rgb(data_root / record.rel_path)
            cached = read_cached_rgb(tmp_root, record.filename)
            smoke_require(full.shape[0] >= full.shape[1], f"{record.filename}: EXIF-corrected image is not portrait")
            smoke_require(abs(full.shape[1] / full.shape[0] - cached.shape[1] / cached.shape[0]) < 0.01, f"{record.filename}: aspect changed")
        again = build_cache(data_root, selected, workers=workers, output_root=tmp_root)
        smoke_require(again.built == 0, "second cache run is not idempotent")
        log(f"smoke: OK - read and cached {len(selected)} images (EXIF orientations {sorted({r.exif_orientation for r in selected})}), idempotent")

        bits = load_dhashes(tmp_root, selected)
        smoke_require(bits.shape == (len(selected), 64), "dHash matrix has the wrong shape")
        smoke_require(int(np.trace(hamming_matrix(bits))) == 0, "dHash distance to itself is not 0")
        log("smoke: OK - dHash computed")

        fake = synthetic_records()
        names = {r.filename for r in fake}
        for seed in SPLIT_SEEDS:
            split = make_split(fake, None, seed)
            parts = [set(split[k]) for k in ("train", "val", "test")]
            smoke_require(sum(len(p) for p in parts) == len(names) and set.union(*parts) == names, "split does not partition")
            smoke_require(split == make_split(fake, None, seed), "split is not deterministic")
        log("smoke: OK - splits are deterministic partitions")

        write_dataset_counts(tmp_root, selected)
        save_class_domain_counts(tmp_root, selected)
        save_dataset_grid(tmp_root, selected)
        smoke_require(
            all((figures_dir(tmp_root) / f).stat().st_size > 1000 for f in ("class_domain_counts.png", "dataset_grid.png")),
            "figures were not written",
        )
        log("smoke: OK - tables and figures")